In [1]:
# %% 1. Imports
import sys                                               # path bootstrap
import dataclasses                                       # dataclasses.replace for raw updates
from pathlib import Path                                 # repo root discovery

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

# %% 1. Imports
from gbp.loaders import DataLoaderGraph, DataLoaderMockMinimal
from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    OrganicFlowPhase, DemandPhase, Environment, EnvironmentConfig,
)

# %% 2. User config — единственная ручка
mock_config = {"n_stations": 8, "n_trips": 200, "n_days": 3, "num_resources":3, "resource_capacity":100}

# %% 3. Source → Raw → Resolved
import pandas as pd

from gbp.build.pipeline import build_model
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 168,  # 7 days x 24 hours
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}

mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved = build_model(raw)

# %% 4. Simulation
simulation_log = Environment(
    resolved,
    EnvironmentConfig(phases=[OrganicFlowPhase()], seed=42, scenario_id="run"),
).run()
log_tables = simulation_log.to_dataframes()

2026-04-26 13:28:24 [info     ] load_start                     loader=graph_core
2026-04-26 13:28:25 [debug    ] source_validated               loader=graph_core
2026-04-26 13:28:26 [debug    ] observed_flow_built            loader=graph_core rows=727
2026-04-26 13:28:26 [debug    ] observed_inventory_built       loader=graph_core rows=140
2026-04-26 13:28:26 [info     ] load_done                      facilities=12 loader=graph_core


In [2]:
mock

DataLoaderMock(tables=12, total_rows=3055)
  stations: df_stations=10, df_station_capacities=20, df_station_costs=10
  depots: df_depots=2, df_depot_capacities=4, df_depot_costs=2
  resources: df_resources=3, df_resource_capacities=3, df_truck_rates=3
  observations: df_inventory_ts=168, df_telemetry_ts=1680, df_trips=1150
  temporal: timestamps=168 (2025-01-01 00:00:00..2025-01-07 23:00:00, freq=<Hour>)

In [38]:
raw

RawModelData(facilities=   facility_id facility_type        name        lat        lon
0      depot_1         depot     depot_1  40.820531 -73.956064
1      depot_2         depot     depot_2  40.762605 -74.011826
2    station_1       station   station_1  40.814057 -73.973170
3    station_2       station   station_2  40.854152 -73.921752
4    station_3       station   station_3  40.738649 -73.938966
5    station_4       station   station_4  40.746683 -73.989392
6    station_5       station   station_5  40.764520 -73.921806
7    station_6       station   station_6  40.714105 -73.925381
8    station_7       station   station_7  40.703386 -73.979628
9    station_8       station   station_8  40.765627 -73.992517
10   station_9       station   station_9  40.720844 -73.941276
11  station_10       station  station_10  40.800567 -74.011832, commodity_categories=  commodity_category_id           name  unit
0          classic_bike   Classic bike  unit
1         electric_bike  Electric bike  unit, resource_categories=  resource_category_id               name  base_capacity
0    rebalancing_truck  Rebalancing truck          100.0, commodities=    commodity_id commodity_category description
0   classic_bike       classic_bike            
1  electric_bike      electric_bike            , resources=  resource_id  resource_category home_facility_id  capacity_override  \
0     truck_1  rebalancing_truck          depot_1              100.0   
1     truck_2  rebalancing_truck          depot_1              100.0   
2     truck_3  rebalancing_truck          depot_1              100.0   

  description  
0        None  
1        None  
2        None  , planning_horizon=  planning_horizon_id          name  start_date    end_date
0                  h1  mock_horizon  2025-01-01  2025-01-08, planning_horizon_segments=  planning_horizon_id  segment_index  start_date    end_date period_type
0                  h1              0  2025-01-01  2025-01-08         day, periods=None, facility_roles=None, facility_operations=   facility_id operation_type  enabled
0      depot_1      receiving     True
1      depot_1        storage     True
2      depot_1       dispatch     True
3      depot_2      receiving     True
4      depot_2        storage     True
5      depot_2       dispatch     True
6    station_1      receiving     True
7    station_1        storage     True
8    station_1       dispatch     True
9    station_2      receiving     True
10   station_2        storage     True
11   station_2       dispatch     True
12   station_3      receiving     True
13   station_3        storage     True
14   station_3       dispatch     True
15   station_4      receiving     True
16   station_4        storage     True
17   station_4       dispatch     True
18   station_5      receiving     True
19   station_5        storage     True
20   station_5       dispatch     True
21   station_6      receiving     True
22   station_6        storage     True
23   station_6       dispatch     True
24   station_7      receiving     True
25   station_7        storage     True
26   station_7       dispatch     True
27   station_8      receiving     True
28   station_8        storage     True
29   station_8       dispatch     True
30   station_9      receiving     True
31   station_9        storage     True
32   station_9       dispatch     True
33  station_10      receiving     True
34  station_10        storage     True
35  station_10       dispatch     True, facility_availability=None, edge_rules=  source_type target_type commodity_category modal_type  enabled
0     station     station       classic_bike       road     True
1     station     station      electric_bike       road     True
2       depot     station       classic_bike       road     True
3       depot     station      electric_bike       road     True
4     station       depot       classic_bike       road     True
5     station       depot      electric_bike       road     True
6       depot       depot       classic

In [37]:
mock.df_truck_rates

,resource_id,cost_per_km,cost_per_hour,fixed_dispatch_cost
0,truck_1,2.38,39.19,123.30
1,truck_2,2.37,42.64,154.41
2,truck_3,1.96,37.93,175.59


In [27]:
resolved.resource_spines['group_0']

,resource_category,name,base_capacity,resource_base_capacity,facility_id,resource_id,resource_cost_per_hour,resource_cost_per_km,resource_fixed_dispatch
0,rebalancing_truck,Rebalancing truck,100.0,100.0,depot_1,truck_1,39.19,2.38,123.30
1,rebalancing_truck,Rebalancing truck,100.0,100.0,depot_1,truck_2,42.64,2.37,154.41
2,rebalancing_truck,Rebalancing truck,100.0,100.0,depot_1,truck_3,37.93,1.96,175.59


In [21]:
log_tables['simulation_flow_log']

,period_index,period_id,phase_name,source_id,target_id,commodity_category,modal_type,quantity,resource_id
0,0,p0,ORGANIC_FLOW,station_1,station_10,classic_bike,None,2.0,None
1,0,p0,ORGANIC_FLOW,station_1,station_2,classic_bike,None,1.0,None
2,0,p0,ORGANIC_FLOW,station_1,station_2,electric_bike,None,1.0,None
3,0,p0,ORGANIC_FLOW,station_1,station_3,classic_bike,None,2.0,None
4,0,p0,ORGANIC_FLOW,station_1,station_4,classic_bike,None,3.0,None
...,...,...,...,...,...,...,...,...,...
722,6,p6,ORGANIC_FLOW,station_9,station_4,classic_bike,None,1.0,None
723,6,p6,ORGANIC_FLOW,station_9,station_5,classic_bike,None,2.0,None
724,6,p6,ORGANIC_FLOW,station_9,station_5,electric_bike,None,2.0,None
725,6,p6,ORGANIC_FLOW,station_9,station_6,electric_bike,None,1.0,None


In [ ]:
# I think it logs we have trips, inventary, mistakes, demand, supply
# So mistakes its simulation_rejected_dispatches_log
# We don't have supply, but we have simulation_unmet_demand_log.
# What is the difference between demand and unment demand?
# Also i have simulation_resource_log
# I my case i don't have resources. But i think i am generating them with default since they are missed.
# Ok. simulation resources just an empty table


In [3]:
ll = DataLoaderGraph(mock)

temporal = ll._build_temporal()
entities = ll._build_entities()
behavior = ll._build_behavior(entities)
distance_data = ll._build_distance_matrix(entities) if ll._config.build_edges else {}
resources = ll._build_resources(entities)

In [12]:
temporal['planning_horizon']

,planning_horizon_id,name,start_date,end_date
0,h1,mock_horizon,2025-01-01,2025-01-08


In [18]:
behavior.keys()

dict_keys(['facility_operations', 'edge_rules'])

In [19]:
behavior['facility_operations']

,facility_id,operation_type,enabled
0,depot_1,receiving,True
1,depot_1,storage,True
2,depot_1,dispatch,True
3,depot_2,receiving,True
4,depot_2,storage,True
5,depot_2,dispatch,True
6,station_1,receiving,True
7,station_1,storage,True
8,station_1,dispatch,True
9,station_2,receiving,True


In [4]:
mock_loader.df_resource_capacities

,resource_id,capacity
0,truck_1,100
1,truck_2,100
2,truck_3,100


In [6]:
log_tables.keys()

dict_keys(['simulation_inventory_log', 'simulation_flow_log', 'simulation_resource_log', 'simulation_unmet_demand_log', 'simulation_rejected_dispatches_log'])

In [2]:
log_tables['simulation_resource_log']

,period_index,period_id,resource_id,resource_category,current_facility_id,status,available_at_period,home_facility_id


In [9]:
log_tables['simulation_resource_log']

,period_index,period_id,resource_id,resource_category,current_facility_id,status,available_at_period,home_facility_id


In [22]:
log_tables['simulation_inventory_log']

,period_index,period_id,facility_id,commodity_category,quantity
0,0,p0,4000.01,classic_bike,4.0
1,0,p0,4000.01,electric_bike,4.0
2,0,p0,4013.02,classic_bike,3.0
3,0,p0,4013.02,electric_bike,2.0
4,0,p0,4026.03,classic_bike,4.0
5,0,p0,4026.03,electric_bike,7.0
6,0,p0,4039.04,classic_bike,2.0
7,0,p0,4039.04,electric_bike,5.0
8,0,p0,4052.05,classic_bike,7.0
9,0,p0,4052.05,electric_bike,3.0
